EDA 2: Measure-Level Performance Analysis

In [1]:
# Objective

# To identify the worst and the best performing healthcare measures across all hospitals.

# This helps highlight system-wide issues that need improvement.

# Questions addressed through this EDA:

# 1- Which measures perform the worst overall?
# 2- Which areas of healthcare need the most attention?

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [4]:
# Loading data
df = pd.read_csv("../data/processed/cleaned_master_data.csv")
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (4480288, 9)


,value,data_period,hospital_name,state,hospital_type,measure_code,measure_name,unit,reported_measure_name
0,108.00,2022-23,Byron Central Hospital,New South Wales,Local Hospital Network,Myh0006,Number Of Elective Surgeries,Surgeries,Urgent Elective Surgery
1,25.00,2022-23,Royal Rehabilitation Hospital,New South Wales,Local Hospital Network,Myh0006,Number Of Elective Surgeries,Surgeries,Laryngectomy
2,75.00,2022-23,Wingham Hospital,New South Wales,Local Hospital Network,Myh0007,Percentage Of Patients Who Waited Longer Than ...,Percent,Curettage And Evacuation Of Uterus
3,1620.00,2022-23,Launceston General Hospital,Tasmania,Local Hospital Network,Myh0007,Percentage Of Patients Who Waited Longer Than ...,Percent,Common Peroneal Nerve Release
4,85.16,2022-23,South West Healthcare [Camperdown],Victoria,Local Hospital Network,Myh0009,Median Waiting Time For Elective Surgery,Days,Procedure For Strabismus (Squint Repair)


In [5]:
# Fetching measures
df['measure_name'].nunique(), df['measure_name'].unique()

(31,
 <StringArray>
 [                                                           'Number Of Elective Surgeries',
              'Percentage Of Patients Who Waited Longer Than 365 Days For Elective Surgery',
                                                 'Median Waiting Time For Elective Surgery',
                                'Number Of Patients Presenting To The Emergency Department',
                                                                 'Number Of Hospital Stays',
                                                         'Number Of Admissions To Hospital',
                                                      'Number Of Specialised Service Units',
                                         'Percentage Of Hospital Stays That Were Overnight',
                                                 'Number Of Surgeries For Malignant Cancer',
                                     'Median Waiting Time For Surgery For Malignant Cancer',
                                                  

In [ ]:
# Abbort: This approach doesn't work for our dataset because it does not have a defined column for unit
# df_percent=df[df['unit']=='percent']
# print("Filetered dataset shape:", df_percent.shape)
# df_percent.head()

Filetered dataset shape: (0, 9)


,value,data_period,hospital_name,state,hospital_type,measure_code,measure_name,unit,reported_measure_name


Creating Measure Categories

In [14]:
def classify_measure(measure_name):
  name = measure_name.lower()

  if "percentage" in name or "rate" in name:
    return "percentage"
  elif "time" in name or "waiting" in name:
    return "time"
  elif "cost" in name:
    return "cost"
  elif "number" in name:
    return "volume"
  else:
    return "other"

# Apply classification
df['measure_category']=df['measure_name'].apply(classify_measure)

df['measure_category'].value_counts()

measure_category
volume        1868778
percentage    1338131
time          1137651
other          122811
cost            12917
Name: count, dtype: int64

Adding Performance Directions

In [ ]:
# Performance Direction Note
# Percentage and rate-based measures → higher values indicate better performance  
# Time, cost, and waiting-related measures → lower values indicate better performance  
# For this analysis, only percentage-based measures are considered, so performance interpretation is consistent (higher = better).
# However, defining performance direction becomes essential in future analyses involving time,
#  cost, or mixed metric types to ensure correct comparisons and interpretations.

def performance_direction(measure_name):
  name= measure_name.lower()

  if "percentage" in name or "rate" in name:
    return "higher_better"
  elif "time" in name or "waiting" in name or "cost" in name:
    return "lower_better"
  else:
    return "neutral"
  
df['performance_direction']=df['measure_name'].apply(performance_direction)
df[['measure_name','measure_category','performance_direction']].drop_duplicates().head(10)

,measure_name,measure_category,performance_direction
0,Number Of Elective Surgeries,volume,neutral
2,Percentage Of Patients Who Waited Longer Than ...,percentage,higher_better
4,Median Waiting Time For Elective Surgery,time,lower_better
12,Number Of Patients Presenting To The Emergency...,volume,neutral
17,Number Of Hospital Stays,volume,neutral
33,Number Of Admissions To Hospital,volume,neutral
35,Number Of Specialised Service Units,volume,neutral
50,Percentage Of Hospital Stays That Were Overnight,percentage,higher_better
63,Number Of Surgeries For Malignant Cancer,volume,neutral
75,Median Waiting Time For Surgery For Malignant ...,time,lower_better


In [19]:
# Focusing on percentage measures for performance analysis
df_percent =df[df['measure_category']=='percentage']
print("Shape:", df_percent.shape)
print(df.head())

Shape: (1338131, 11)
     value data_period                       hospital_name            state  \
0   108.00     2022-23              Byron Central Hospital  New South Wales   
1    25.00     2022-23       Royal Rehabilitation Hospital  New South Wales   
2    75.00     2022-23                    Wingham Hospital  New South Wales   
3  1620.00     2022-23         Launceston General Hospital         Tasmania   
4    85.16     2022-23  South West Healthcare [Camperdown]         Victoria   

            hospital_type measure_code  \
0  Local Hospital Network      Myh0006   
1  Local Hospital Network      Myh0006   
2  Local Hospital Network      Myh0007   
3  Local Hospital Network      Myh0007   
4  Local Hospital Network      Myh0009   

                                        measure_name       unit  \
0                       Number Of Elective Surgeries  Surgeries   
1                       Number Of Elective Surgeries  Surgeries   
2  Percentage Of Patients Who Waited Longer Than .

Aggeregating Measure Performance

In [21]:
measure_perf =(
  df_percent.groupby('measure_name')['value'].mean().reset_index().sort_values(by='value', ascending=True
))
measure_perf.head(10)
# note: low values are worse for percentage measures, so we sort in ascending order to identify worst performers at the top.

,measure_name,value
6,Percentage Of People Who Received Surgery For ...,183.921634
9,Rate Of Healthcare-Associated Staphylococcus A...,197.566346
3,Percentage Of Patients Who Depart The Emergenc...,198.382137
5,Percentage Of Patients Who Waited Longer Than ...,201.023728
1,Percentage Of Hospital Stays That Were Overnight,202.064932
0,Hand Hygiene Rate,203.061027
7,Percentage Of People Who Received Surgery For ...,203.082917
2,Percentage Of Patients Who Commenced Treatment...,209.713173
4,Percentage Of Patients Who Received Their Surg...,210.786732
8,Percentage Of Private Patients,238.261405
